In [1]:
import numpy as np
import random

In [2]:
class Value:
    def __init__(self, value, children = (), op = '', label = '', grad = 0):
        
        self.value = value
        self._prev = children
        self._backward = lambda : None
        self.grad = grad
        self._op = op
        self.label = label

    def __repr__(self):
        
        return f'{self.value}'

    def __add__(self, other):

        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(self.value + other.value, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad 
        out._backward = _backward
        
        return out

    def __sub__(self, other):

        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(self.value - other.value, (self, other), '-')
        
        def _backward():
             self.grad += 1.0 * out.grad
             other.grad += -1.0 * out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):

        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(self.value * other.value, (self, other), '*')
        
        def _backward():
             self.grad += out.grad * other.value
             other.grad += out.grad * self.value
        out._backward = _backward
        
        return out

    def __truediv__(self, other):
        
        other = other if isinstance(other, Value) else Value(other)

        if other.value == 0:
            raise ZeroDivisionError('Division with zero is not allowed')
        
        out = Value(self.value / other.value, (self, other), '/')
        
        def _backward():
             self.grad += out.grad * (other.value**-1)
             other.grad += out.grad * (self.value * (-1 / other.value**2))
        out._backward = _backward
        
        return out

    def tanh(self):
        x = self.value
        t = np.tanh(self.value)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += out.grad * (1.0 - t**2)
        out._backward = _backward
        
        return out

    def backward(self):
        
        topo = []
        visited = set()
        
        def dfs(node):
    
            if node in visited:
                return
    
            visited.add(node)
    
            for child in node._prev:
                dfs(child)
    
            topo.append(node)
    
        dfs(self)
    
        self.grad = 1

        for node in reversed(topo):
            node._backward()

In [3]:
class Neuron:

    def __init__(self, n):
        self.w = [Value(random.uniform(-1,1)) for _ in range(n)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        aggr = sum((xi*wi for xi, wi in zip(x, self.w)), self.b)
        activated = aggr.tanh()
        return activated

    def __repr__(self):
        return f'Weights: {[w for w in self.w]}, Bias: {self.b}'

In [4]:
class Layer:

    def __init__(self, inp, out):
        self.inp = inp
        self.out = out
        self.neurons = [Neuron(self.inp) for _ in range(self.out)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out

    def __repr__(self):
        out_string = [f"Neuron : {neuron}" for neuron in self.neurons]
        return ',\n'.join(out_string)

In [5]:
class MLP:

    def __init__(self, ins, outs):
        sizes = [ins] + outs
        self.layers = [Layer(sizes[i-1], sizes[i]) for i in range(1, len(sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)

        return x

    def __repr__(self):
        out_string = [f'Layer{i}:\n{layer}' for i, layer in enumerate(self.layers, start = 1)]
        return ',\n\n'.join(out_string)

    def zero_grad(self):
        for layer in self.layers:
            for neuron in layer.neurons:
                neuron.b.grad = 0
                for w in neuron.w:
                    w.grad = 0

    def update_weights(self, lr = 0.01):
        for layer in self.layers:
            for neuron in layer.neurons:
                neuron.b.value -= lr * neuron.b.grad
                for w in neuron.w:
                    w.value -= lr * w.grad

In [6]:
def MSE(actual, pred):

    if len(actual) != len(pred):
        raise ValueError('Shape mismatch')
    
    err_sq_sum = sum(((a-p) * (a-p) for a,p in zip(actual, pred)), Value(0))
    return err_sq_sum / len(actual)

In [7]:
model = MLP(2, [4, 4, 1])

xs = [
    [Value(0.0), Value(0.0)],
    [Value(0.0), Value(1.0)],
    [Value(1.0), Value(0.0)],
    [Value(1.0), Value(1.0)],
]

ys = [
    Value(0.0),
    Value(1.0),
    Value(1.0),
    Value(0.0),
]

for epoch in range(200):
    preds = [model(x)[0] for x in xs]
    loss = MSE(ys, preds)

    model.zero_grad()
    loss.backward()
    model.update_weights(lr=0.1)

    if epoch % 10 == 0:
        print(epoch, loss.value)

0 0.4834203826112101
10 0.47772022615369447
20 0.4670581585157113
30 0.44255622681618656
40 0.3713112750996121
50 0.26490958635159323
60 0.23264094446170394
70 0.21830124183097996
80 0.20715284388338942
90 0.19749545794086995
100 0.18925804234888524
110 0.1823473367257088
120 0.17654913553229784
130 0.1716199353043555
140 0.1673481354131397
150 0.16356990985451872
160 0.16016205890145216
170 0.15702887949794875
180 0.15408985328866
190 0.15126904709785843


In [8]:
for x, y in zip(xs, ys):
    pred = model(x)[0]
    print([v.value for v in x], "pred:", pred.value, "target:", y.value)

[0.0, 0.0] pred: 0.543033337634777 target: 0.0
[0.0, 1.0] pred: 0.7957040004120756 target: 1.0
[1.0, 0.0] pred: 0.49558605127849764 target: 1.0
[1.0, 1.0] pred: 0.05369682899268975 target: 0.0
